In [ ]:
# ===========================================
# Minimal-5 vs Minimal-5+Embedding — Team TEST 전용 오프라인 평가
#   - 학습: eval_set == 'prior' (1~5회차)
#   - 평가: eval_set == 'test'  (팀 내부 TEST, 라벨 포함)
#   - eval_set == 'train'(6th)는 완전히 제외
#   - DMatrix 열수/이름 엄격 정합(중복 컬럼 제거 + 훈련 피처 그대로 사용)
# ===========================================
import os, json, re, time, subprocess, gc, warnings
warnings.filterwarnings("ignore", category=UserWarning)

# ===== 0) 환경/경로 =====
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Drive mount skipped (non-Colab env):", e)

DATA_DIR    = "/content/drive/MyDrive/data/instacart"
MASTER_PATH = f"{DATA_DIR}/master_dataset_with_roles_final.csv"
ADDRESS_PATH= f"{DATA_DIR}/address(in).csv"     # user_id 순서가 embedding.npy와 동일
EMBED_PATH  = f"{DATA_DIR}/embedding.npy"       # (n_users, 128)

OUT_DIR     = f"{DATA_DIR}/perf_min5_vs_min5plusEmb_TEAMTEST_only"
os.makedirs(OUT_DIR, exist_ok=True)

# 산출물
OUT_SUMMARY_CSV  = f"{OUT_DIR}/summary.csv"
OUT_SUMMARY_JSON = f"{OUT_DIR}/summary.json"
OUT_PER_ORDER    = f"{OUT_DIR}/per_order_f1_teamTEST.csv"
OUT_MIN_BETTER50 = f"{OUT_DIR}/min5_better_top50_teamTEST.csv"
OUT_EMB_BETTER50 = f"{OUT_DIR}/min5plusEmb_better_top50_teamTEST.csv"
OUT_OOF_MIN      = f"{OUT_DIR}/oof_min5.csv"
OUT_OOF_MIN_EMB  = f"{OUT_DIR}/oof_min5plusEmb.csv"
OUT_DET_MIN_TT   = f"{OUT_DIR}/detailed_min5_teamTEST.csv"
OUT_DET_EMB_TT   = f"{OUT_DIR}/detailed_min5plusEmb_teamTEST.csv"
OUT_ORD_MIN_TT   = None
OUT_ORD_EMB_TT   = None
OUT_FI_MIN       = f"{OUT_DIR}/fi_min5.csv"
OUT_FI_MIN_EMB   = f"{OUT_DIR}/fi_min5plusEmb.csv"

# ===== 1) 유틸 =====
import numpy as np
import pandas as pd
from typing import List, Optional, Dict, Tuple
from dataclasses import dataclass
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, log_loss
import xgboost as xgb

def first_existing(cands: List[str], cols: List[str]) -> Optional[str]:
    for c in cands:
        if c in cols: return c
    return None

def unique_keep_order(seq: List[str]) -> List[str]:
    # 순서 유지 중복 제거
    return list(dict.fromkeys(seq))

def drop_duplicated_columns(df: pd.DataFrame) -> pd.DataFrame:
    if df.columns.duplicated().any():
        before = len(df.columns)
        df = df.loc[:, ~df.columns.duplicated(keep='first')].copy()
        after = len(df.columns)
        print(f"[CLEAN] drop duplicated columns: {before} -> {after}")
    return df

def is_binary_series(s: pd.Series) -> bool:
    vals = pd.unique(s.dropna())
    return set(vals.tolist()).issubset({0,1}) or set(vals.tolist()).issubset({0.0,1.0})

# Instacart 주문단위 F1 (빈 예측·정답 → 'None' 규칙)
def f1_single(true_set: set, pred_set: set) -> float:
    if len(true_set) == 0 and len(pred_set) == 0:
        return 1.0
    if len(pred_set) == 0:
        pred_set = {"None"}
    if len(true_set) == 0:
        true_set = {"None"}
    tp = len(true_set & pred_set)
    if tp == 0:
        return 0.0
    precision = tp / len(pred_set)
    recall    = tp / len(true_set)
    return 0.0 if (precision+recall)==0 else 2*precision*recall/(precision+recall)

def order_level_f1(df: pd.DataFrame, order_col: str, product_col: str,
                   target_col: str, proba_col: str, thr: float) -> float:
    f1s = []
    for _, g in df.groupby(order_col):
        true_set = set(g.loc[g[target_col]==1, product_col].tolist())
        pred_set = set(g.loc[g[proba_col]>=thr, product_col].tolist())
        f1s.append(f1_single(true_set, pred_set))
    return float(np.mean(f1s)) if len(f1s) else 0.0

def search_best_threshold_fast(
    df: pd.DataFrame, order_col: str, product_col: str,
    target_col: str, proba_col: str,
    quantile_lo: float = 0.05, quantile_hi: float = 0.95,
    n_candidates: int = 31, max_orders: int = 50_000,
    seed: int = 2025, verbose: bool = True
) -> Tuple[float, float]:
    t0 = time.time()
    orders = df[order_col].drop_duplicates()
    if (max_orders is not None) and (len(orders) > max_orders):
        sampled_orders = orders.sample(max_orders, random_state=seed)
        sub = df[df[order_col].isin(sampled_orders)][[order_col, product_col, target_col, proba_col]].copy()
        if verbose: print(f"[thr-search] sample orders: {len(sampled_orders):,}  rows: {len(sub):,}")
    else:
        sub = df[[order_col, product_col, target_col, proba_col]].copy()
        if verbose: print(f"[thr-search] full orders: {sub[order_col].nunique():,}  rows: {len(sub):,}")

    qs = np.linspace(quantile_lo, quantile_hi, n_candidates)
    thr_list = np.unique(sub[proba_col].quantile(qs).values)
    if verbose: print(f"[thr-search] candidates: {len(thr_list)}  (q {quantile_lo:.2f}~{quantile_hi:.2f})")

    true_cnt = sub.groupby(order_col)[target_col].sum().astype(np.int32)
    best_thr, best_f1 = 0.5, -1.0
    for i, t in enumerate(thr_list, 1):
        mask    = (sub[proba_col] >= t)
        pred_cnt= sub.loc[mask].groupby(order_col, observed=True)[proba_col].size()
        tp_cnt  = sub.loc[mask & (sub[target_col] == 1)].groupby(order_col, observed=True)[target_col].size()
        agg = pd.DataFrame({
            'true': true_cnt,
            'pred': pred_cnt.reindex(true_cnt.index, fill_value=0).astype(np.int32),
            'tp'  : tp_cnt.reindex(true_cnt.index,  fill_value=0).astype(np.int32),
        })
        true = agg['true'].values
        pred = agg['pred'].values
        tp   = agg['tp'].values
        none_case = (pred == 0) & (true == 0)
        with np.errstate(divide='ignore', invalid='ignore'):
            precision = np.divide(tp, pred, out=np.zeros_like(tp, dtype=float), where=pred>0)
            recall    = np.divide(tp, true, out=np.zeros_like(tp, dtype=float), where=true>0)
            denom     = precision + recall
            f1_arr    = np.divide(2*precision*recall, denom, out=np.zeros_like(denom), where=denom>0)
        f1_arr[none_case] = 1.0
        f1 = float(f1_arr.mean())
        if f1 > best_f1:
            best_f1, best_thr = f1, float(t)
        if verbose and (i % max(1, len(thr_list)//5) == 0 or i == len(thr_list)):
            print(f"[thr-search] {i}/{len(thr_list)}  thr={t:.4f}  F1={f1:.5f}")
    if verbose: print(f"[thr-search] best_thr={best_thr:.4f}  best_f1={best_f1:.5f}  total={time.time()-t0:.1f}s")
    return best_thr, best_f1

def fast_auc_logloss(df: pd.DataFrame, y_col: str, p_col: str,
                     max_rows: int = 2_000_000, seed: int = 2025):
    y = df[y_col].to_numpy()
    p = np.clip(df[p_col].to_numpy(dtype=np.float64), 1e-15, 1-1e-15)
    if len(y) > max_rows:
        df = df.sample(n=max_rows, random_state=seed)
        y = df[y_col].to_numpy()
        p = np.clip(df[p_col].to_numpy(dtype=np.float64), 1e-15, 1-1e-15)
        print(f"[metrics] sampled {len(y):,} rows for AUC/Logloss")
    auc = roc_auc_score(y, p) if len(np.unique(y))>1 else np.nan
    ll  = float(log_loss(y, p))
    return auc, ll

def _bst_predict_proba(bst: xgb.Booster, dmat: xgb.DMatrix) -> np.ndarray:
    bi = getattr(bst, "best_iteration", None)
    if bi is not None:
        try:
            return bst.predict(dmat, iteration_range=(0, int(bi)+1)).astype(np.float32)
        except TypeError:
            pass
        try:
            return bst.predict(dmat, ntree_limit=getattr(bst, "best_ntree_limit", int(bi)+1)).astype(np.float32)
        except Exception:
            pass
    return bst.predict(dmat).astype(np.float32)

def detect_device_and_tree_method():
    try:
        _ = subprocess.check_output(["nvidia-smi"])
        return "cuda", "hist"
    except Exception:
        return "cpu", "hist"

DEVICE, TREE_METHOD = detect_device_and_tree_method()
print(f"[INFO] device={DEVICE}  tree_method={TREE_METHOD}  xgboost={xgb.__version__}")

# ===== 2) 데이터 로드/분리 (train 6th 제외) =====
df = pd.read_csv(MASTER_PATH, low_memory=False)
df = drop_duplicated_columns(df)

cols        = df.columns.tolist()
target_col  = first_existing(['reordered','label','target','y','is_reordered'], cols)
product_col = first_existing(['product_id','pid','product'], cols)
order_col   = first_existing(['order_id','oid','order'], cols)
member_col  = first_existing(['user_id','member_id','uid','user'], cols)
eval_col    = 'eval_set' if 'eval_set' in cols else None
assert target_col and product_col and (order_col or member_col) and eval_col, "필수 컬럼 누락"

ev = df[eval_col].astype(str).str.lower()
mask_prior = ev.eq('prior')   # 학습(1~5)
mask_team  = ev.eq('test')    # 팀 내부 TEST(라벨 포함)
df_prior   = df.loc[mask_prior].copy()
df_teamT   = df.loc[mask_team].copy()

# 라벨 이진 정리
if not is_binary_series(df_prior[target_col]):
    df_prior[target_col] = (df_prior[target_col] > 0).astype(np.int8)
if (target_col in df_teamT.columns) and (not is_binary_series(df_teamT[target_col])):
    df_teamT[target_col] = (df_teamT[target_col] > 0).astype(np.int8)

order_key = order_col if order_col else member_col
group_key = member_col if member_col else order_col
print(f"[INFO] rows — prior(train)={len(df_prior):,}, teamTEST={len(df_teamT):,}")

# ===== 3) Minimal-5 피처 =====
hour_col = first_existing(['order_hour_of_day','hour','order_hour','hour_of_day'], df.columns.tolist())
aisle_col= first_existing(['aisle_id','aisle'], df.columns.tolist())
dept_col = first_existing(['department_id','dept_id','department'], df.columns.tolist())
assert hour_col and aisle_col and dept_col and product_col and member_col, "미니멀 5 피처 누락"

uid_num_col = f"{member_col}_num"
for d in (df_prior, df_teamT):
    d[uid_num_col] = pd.to_numeric(d[member_col], errors="coerce").astype('float32')

MIN_FEATS = [hour_col, aisle_col, dept_col, product_col, uid_num_col]
print("[INFO] minimal-5 features:", MIN_FEATS)

for d in (df_prior, df_teamT):
    for c in [hour_col, aisle_col, dept_col, product_col]:
        d[c] = pd.to_numeric(d[c], errors="coerce").astype('float32')
    d[MIN_FEATS] = d[MIN_FEATS].fillna(-1.0)

# ===== 4) 임베딩 조인 =====
addr = pd.read_csv(ADDRESS_PATH)
user_ids_in_order = addr['user_id'].astype(str).tolist()
seen=set(); ordered_unique=[]
for u in user_ids_in_order:
    if u not in seen:
        seen.add(u); ordered_unique.append(u)

emb = np.load(EMBED_PATH)  # (n_users, d)
assert emb.shape[0] == len(ordered_unique), f"임베딩 행수({emb.shape[0]}) != user_id 수({len(ordered_unique)})"
emb_dim  = emb.shape[1]
emb_cols = [f"emb_{i}" for i in range(emb_dim)]
df_emb   = pd.DataFrame(emb, columns=emb_cols)
df_emb.insert(0, member_col, ordered_unique)

def join_emb(d: pd.DataFrame) -> pd.DataFrame:
    dd = d.copy()
    dd[member_col] = dd[member_col].astype(np.int64).astype(str)
    dd = dd.merge(df_emb, on=member_col, how='left', validate='many_to_one')
    for c in emb_cols: dd[c] = dd[c].astype('float32')
    miss = dd[emb_cols].isna().any(axis=1).sum()
    if miss:
        print(f"[WARN] missing embedding — rows {miss:,} — filling 0")
        dd[emb_cols] = dd[emb_cols].fillna(0.0)
    return dd

df_prior_emb = join_emb(df_prior)
df_teamT_emb = join_emb(df_teamT)

# ===== 5) 학습 표본(동일) 구성 =====
NEG_POS_RATIO  = 5.0
MAX_TRAIN_ROWS = 3_000_000
RANDOM_SEED    = 2025
np.random.seed(RANDOM_SEED)

pos_mask = (df_prior[target_col] == 1)
neg_mask = ~pos_mask
n_pos    = int(pos_mask.sum())
max_neg  = int(min(neg_mask.sum(), NEG_POS_RATIO * n_pos))
neg_idx  = df_prior.loc[neg_mask].sample(n=max_neg, random_state=RANDOM_SEED).index if max_neg>0 else df_prior.loc[neg_mask].index
train_idx= df_prior.loc[pos_mask].index.union(neg_idx)
if len(train_idx) > MAX_TRAIN_ROWS:
    train_idx = pd.Index(np.random.choice(train_idx, size=MAX_TRAIN_ROWS, replace=False))

df_tr_min     = df_prior.loc[train_idx].copy()
df_tr_min_emb = df_prior_emb.loc[train_idx].copy()
print(f"[INFO] train sample: {len(df_tr_min):,} rows (pos={int((df_tr_min[target_col]==1).sum()):,})")

# ===== 6) XGBoost 학습/예측 =====
@dataclass
class XGBCfg:
    n_estimators: int = 600
    max_depth: int = 7
    learning_rate: float = 0.05
    subsample: float = 0.90
    colsample_bytree: float = 0.90
    min_child_weight: float = 1.0
    reg_lambda: float = 1.0
    gamma: float = 0.0
    tree_method: str = TREE_METHOD
    device: str = DEVICE
    random_state: int = 42

cfg = XGBCfg()
NFOLDS = 3
EARLY_STOP = 50

def fit_xgb_cv_booster(df_tr: pd.DataFrame, features: List[str], target: str,
                       groups: Optional[pd.Series], cfg: XGBCfg,
                       model_prefix: str, n_splits: int = NFOLDS,
                       early_stopping_rounds: int = EARLY_STOP) -> Dict:
    X = df_tr[features].values.astype(np.float32)
    y = df_tr[target].values.astype(np.float32)
    if groups is None:
        groups = np.arange(len(y)) % n_splits
    splitter = GroupKFold(n_splits=n_splits)
    params = {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "max_depth": cfg.max_depth,
        "eta": cfg.learning_rate,
        "subsample": cfg.subsample,
        "colsample_bytree": cfg.colsample_bytree,
        "min_child_weight": cfg.min_child_weight,
        "lambda": cfg.reg_lambda,
        "gamma": cfg.gamma,
        "tree_method": "hist",
        "device": cfg.device,
        "verbosity": 1,
        "seed": cfg.random_state,
    }
    oof = np.zeros(len(df_tr), dtype=np.float32)
    models, metrics = [], []
    for fold, (tr_idx, va_idx) in enumerate(splitter.split(X, y, groups=groups), 1):
        dtr = xgb.DMatrix(X[tr_idx], label=y[tr_idx], feature_names=features)
        dva = xgb.DMatrix(X[va_idx], label=y[va_idx], feature_names=features)
        bst = xgb.train(params=params, dtrain=dtr, num_boost_round=cfg.n_estimators,
                        evals=[(dva, "valid")], early_stopping_rounds=early_stopping_rounds,
                        verbose_eval=False)
        proba_va = _bst_predict_proba(bst, dva)
        oof[va_idx] = proba_va
        auc = roc_auc_score(y[va_idx], proba_va) if len(np.unique(y[va_idx]))>1 else np.nan
        ll  = float(log_loss(y[va_idx], np.clip(proba_va,1e-15,1-1e-15)))
        print(f"[{model_prefix}] Fold {fold} | AUC={auc:.5f} | Logloss={ll:.5f} | BestIter={getattr(bst,'best_iteration',None)}")
        bst.save_model(os.path.join(OUT_DIR, f"{model_prefix}_fold{fold}.json"))
        models.append(bst); metrics.append({"fold":fold,"auc":float(auc),"logloss":float(ll)})
        del dtr, dva; gc.collect()
    return {"oof":oof, "models":models, "metrics":metrics, "features":features}

def predict_with_boosters_STRICT(df_in: pd.DataFrame,
                                 boosters: List[xgb.Booster],
                                 trained_feature_names: List[str],
                                 out_col: str,
                                 log_tag: str):
    # 0) 중복 컬럼 제거(핵심: 동일 라벨 열이 2개 있으면 df_in[feat]가 2열을 뽑아 DMatrix mismatch 발생)
    df_in = drop_duplicated_columns(df_in)

    # 1) 훈련 피처가 모두 존재하는지 확인, 없으면 0.0 채움
    for f in trained_feature_names:
        if f not in df_in.columns:
            df_in[f] = 0.0

    # 2) 정확히 '훈련에 사용한 피처'만, '동일한 순서'로 선택
    model_feats = list(trained_feature_names)
    X = df_in.loc[:, model_feats].values.astype(np.float32)

    # 3) sanity check
    if X.shape[1] != len(model_feats):
        raise RuntimeError(f"[{log_tag}] feature shape mismatch: X has {X.shape[1]} cols but model_feats has {len(model_feats)}")

    dmat = xgb.DMatrix(X, feature_names=model_feats)
    preds = np.zeros(len(df_in), dtype=np.float32)
    for bst in boosters:
        preds += _bst_predict_proba(bst, dmat)
    preds /= max(1, len(boosters))
    df_in[out_col] = preds
    print(f"[{log_tag}] predict ok: used {len(model_feats)} features; dmat cols={X.shape[1]}; preds shape={preds.shape}")
    return df_in

def save_importance_booster(models: List[xgb.Booster], features: List[str], out_path: str):
    agg = np.zeros(len(features), dtype=np.float64)
    for bst in models:
        fmap = bst.get_score(importance_type='gain')
        for i, f in enumerate(features):
            agg[i] += float(fmap.get(f, 0.0) or fmap.get(f"f{i}", 0.0) or 0.0)
    agg /= max(1, len(models))
    pd.DataFrame({'feature': features, 'gain': agg}).sort_values('gain', ascending=False).to_csv(out_path, index=False)
    print("Saved feature importance →", out_path)

# ===== 7) 모델 A: Minimal-5 =====
FEATS_MIN = MIN_FEATS[:]  # 방어적 복사
groups = df_tr_min[group_key] if group_key in df_tr_min.columns else None
res_MIN = fit_xgb_cv_booster(df_tr_min, FEATS_MIN, target_col, groups, cfg, "xgb_MIN")

df_tr_min["proba_min"] = res_MIN["oof"]
thr_min, f1_min = search_best_threshold_fast(
    df_tr_min[[order_key, product_col, target_col, "proba_min"]],
    order_key, product_col, target_col, "proba_min",
    n_candidates=31, max_orders=50_000, verbose=True
)
auc_min, ll_min = fast_auc_logloss(df_tr_min, target_col, "proba_min", max_rows=2_000_000)
print(f"[OOF-MIN] BestThr={thr_min:.4f} | F1={f1_min:.5f} | AUC≈{auc_min:.5f} | Logloss≈{ll_min:.5f}")

# ===== 8) 모델 B: Minimal-5 + Embedding =====
FEATS_MIN_EMB = FEATS_MIN + emb_cols
groups = df_tr_min_emb[group_key] if group_key in df_tr_min_emb.columns else None
res_MIN_EMB = fit_xgb_cv_booster(df_tr_min_emb, FEATS_MIN_EMB, target_col, groups, cfg, "xgb_MINpEMB")

df_tr_min_emb["proba_min_emb"] = res_MIN_EMB["oof"]
thr_min_emb, f1_min_emb = search_best_threshold_fast(
    df_tr_min_emb[[order_key, product_col, target_col, "proba_min_emb"]],
    order_key, product_col, target_col, "proba_min_emb",
    n_candidates=31, max_orders=50_000, verbose=True
)
auc_min_emb, ll_min_emb = fast_auc_logloss(df_tr_min_emb, target_col, "proba_min_emb", max_rows=2_000_000)
print(f"[OOF-MIN+EMB] BestThr={thr_min_emb:.4f} | F1={f1_min_emb:.5f} | AUC≈{auc_min_emb:.5f} | Logloss≈{ll_min_emb:.5f}")

# ===== 9) TEAM-TEST 추론/평가 =====
# (필요 컬럼만, 중복 없이 뽑아온 뒤 예측)
tt_cols_min     = unique_keep_order(FEATS_MIN + [order_key, product_col, target_col, member_col])
tt_cols_min_emb = unique_keep_order(FEATS_MIN_EMB + [order_key, product_col, target_col, member_col])

df_tt_min_raw   = drop_duplicated_columns(df_teamT.loc[:, tt_cols_min].copy())
df_tt_emb_raw   = drop_duplicated_columns(df_teamT_emb.loc[:, tt_cols_min_emb].copy())

# Strict alignment 예측 → df에 proba 열 생성
df_tt_min = predict_with_boosters_STRICT(df_tt_min_raw,   res_MIN["models"],     res_MIN["features"],     "proba_min",     "TEAMTEST_MIN")
df_tt_me  = predict_with_boosters_STRICT(df_tt_emb_raw,   res_MIN_EMB["models"], res_MIN_EMB["features"], "proba_min_emb", "TEAMTEST_MIN+EMB")

# 팀TEST 성능(OOF 임계치 고정)
test_auc_min, test_ll_min = fast_auc_logloss(df_tt_min, target_col, "proba_min", max_rows=2_000_000)
test_f1_min  = order_level_f1(df_tt_min, order_key, product_col, target_col, "proba_min", thr_min)
# 진단용: 팀TEST 내 최적 F1
_, test_f1_min_best = search_best_threshold_fast(df_tt_min[[order_key, product_col, target_col, "proba_min"]],
                                                 order_key, product_col, target_col, "proba_min", verbose=False)

test_auc_me, test_ll_me = fast_auc_logloss(df_tt_me, target_col, "proba_min_emb", max_rows=2_000_000)
test_f1_me   = order_level_f1(df_tt_me,  order_key, product_col, target_col, "proba_min_emb", thr_min_emb)
_, test_f1_me_best = search_best_threshold_fast(df_tt_me[[order_key, product_col, target_col, "proba_min_emb"]],
                                                order_key, product_col, target_col, "proba_min_emb", verbose=False)

print(f"[TEAM-TEST] MIN   @thr={thr_min:.4f}  F1={test_f1_min:.5f}  AUC≈{test_auc_min:.5f}  Logloss≈{test_ll_min:.5f}  | diag bestF1={test_f1_min_best:.5f}")
print(f"[TEAM-TEST] MIN+E @thr={thr_min_emb:.4f}  F1={test_f1_me:.5f}  AUC≈{test_auc_me:.5f}  Logloss≈{test_ll_me:.5f}  | diag bestF1={test_f1_me_best:.5f}")

# 주문별 제출 문자열/디테일 저장
def to_pred_string(g: pd.DataFrame, pcol: str, thr: float) -> str:
    items = g.loc[g[pcol] >= thr, product_col].astype(str).tolist()
    return "None" if len(items)==0 else " ".join(items)

sub_tt_min = df_tt_min.groupby(order_key, group_keys=False).apply(lambda g: to_pred_string(g, "proba_min", thr_min)).reset_index()
sub_tt_min.columns = [order_key, "products"]
OUT_ORD_MIN_TT = f"{OUT_DIR}/orders_min5_teamTEST_thr_{thr_min:.2f}.csv"
sub_tt_min.to_csv(OUT_ORD_MIN_TT, index=False)
df_tt_min[[order_key, member_col, product_col, target_col, "proba_min"]].to_csv(OUT_DET_MIN_TT, index=False)

sub_tt_me = df_tt_me.groupby(order_key, group_keys=False).apply(lambda g: to_pred_string(g, "proba_min_emb", thr_min_emb)).reset_index()
sub_tt_me.columns = [order_key, "products"]
OUT_ORD_EMB_TT = f"{OUT_DIR}/orders_min5plusEmb_teamTEST_thr_{thr_min_emb:.2f}.csv"
sub_tt_me.to_csv(OUT_ORD_EMB_TT, index=False)
df_tt_me[[order_key, member_col, product_col, target_col, "proba_min_emb"]].to_csv(OUT_DET_EMB_TT, index=False)

# OOF/Feature importance 저장
df_tr_min[[order_key, member_col, product_col, target_col, "proba_min"]].to_csv(OUT_OOF_MIN, index=False)
df_tr_min_emb[[order_key, member_col, product_col, target_col, "proba_min_emb"]].to_csv(OUT_OOF_MIN_EMB, index=False)
save_importance_booster(res_MIN['models'],     res_MIN['features'],     OUT_FI_MIN)
save_importance_booster(res_MIN_EMB['models'], res_MIN_EMB['features'], OUT_FI_MIN_EMB)

# ===== 10) per-order 비교 및 요약 =====
def _true_sets(df_split):
    tmp = df_split[[order_key, product_col, target_col]].copy()
    tmp[order_key]   = tmp[order_key].astype(str)
    tmp[product_col] = tmp[product_col].astype(str)
    return tmp.loc[tmp[target_col]==1].groupby(order_key)[product_col].agg(lambda s: set(s.tolist())).to_dict()

def _pred_sets(df_split, proba_col, thr):
    out = {}
    for oid, g in df_split.groupby(order_key):
        out[str(oid)] = set(g.loc[g[proba_col]>=thr, product_col].astype(str).tolist())
    return out

true_tt = _true_sets(df_tt_min)
pred_tt_m  = _pred_sets(df_tt_min, "proba_min", thr_min)
pred_tt_me = _pred_sets(df_tt_me,  "proba_min_emb", thr_min_emb)

orders_tt = list(set(true_tt.keys()) & set(pred_tt_m.keys()) & set(pred_tt_me.keys()))
rows = []
for oid in orders_tt:
    ts = true_tt[oid]; ms = pred_tt_m.get(oid,set()); es = pred_tt_me.get(oid,set())
    f1m = f1_single(ts, ms); f1e = f1_single(ts, es)
    rows.append({"order_id": oid, "F1_min5": f1m, "F1_min5plusEmb": f1e,
                 "F1_diff(min5plusEmb - min5)": f1e - f1m, "True_size":len(ts),
                 "Min_pred_size":len(ms), "MinEmb_pred_size":len(es)})
per_order_tt = pd.DataFrame(rows)
if len(per_order_tt):
    per_order_tt.sort_values("F1_diff(min5plusEmb - min5)", ascending=False).head(50).to_csv(OUT_EMB_BETTER50, index=False)
    per_order_tt.sort_values("F1_diff(min5plusEmb - min5)", ascending=True ).head(50).to_csv(OUT_MIN_BETTER50, index=False)
    per_order_tt.to_csv(OUT_PER_ORDER, index=False)

summary_rows = [
    {"split":"OOF(prior)", "model":"XGB_min5",
     "features_used": FEATS_MIN,
     "macro_F1": float(f1_min), "AUC": float(auc_min), "Logloss": float(ll_min),
     "threshold": float(thr_min), "rows": int(len(df_tr_min))},
    {"split":"OOF(prior)", "model":"XGB_min5_plus_embedding",
     "features_used": FEATS_MIN_EMB[:5] + [f"emb_0..emb_{emb_dim-1}"],
     "macro_F1": float(f1_min_emb), "AUC": float(auc_min_emb), "Logloss": float(ll_min_emb),
     "threshold": float(thr_min_emb), "rows": int(len(df_tr_min_emb))},

    {"split":"TEAM-TEST", "model":"XGB_min5",
     "macro_F1": float(test_f1_min), "AUC": float(test_auc_min), "Logloss": float(test_ll_min),
     "threshold": float(thr_min), "diag_bestF1": float(test_f1_min_best), "rows": int(len(df_tt_min))},
    {"split":"TEAM-TEST", "model":"XGB_min5_plus_embedding",
     "macro_F1": float(test_f1_me), "AUC": float(test_auc_me), "Logloss": float(test_ll_me),
     "threshold": float(thr_min_emb), "diag_bestF1": float(test_f1_me_best), "rows": int(len(df_tt_me))}
]
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUT_SUMMARY_CSV, index=False)
with open(OUT_SUMMARY_JSON, "w") as f:
    json.dump({
        "summary": summary_rows,
        "notes": {
            "master": MASTER_PATH,
            "address": ADDRESS_PATH,
            "embedding": EMBED_PATH,
            "order_key": order_key,
            "product_col": product_col,
            "target_col": target_col,
            "group_key_for_cv": group_key,
            "threshold_policy": "OOF-derived threshold applied to teamTEST; per-split bestF1 is diagnostic only"
        },
        "artifacts": {
            "oof_min5": os.path.basename(OUT_OOF_MIN),
            "oof_min5plusEmb": os.path.basename(OUT_OOF_MIN_EMB),
            "detailed_teamTEST_min5": os.path.basename(OUT_DET_MIN_TT),
            "detailed_teamTEST_min5plusEmb": os.path.basename(OUT_DET_EMB_TT),
            "orders_teamTEST_min5": os.path.basename(OUT_ORD_MIN_TT),
            "orders_teamTEST_min5plusEmb": os.path.basename(OUT_ORD_EMB_TT),
            "per_order_cmp_teamTEST": os.path.basename(OUT_PER_ORDER),
            "fi_min5": os.path.basename(OUT_FI_MIN),
            "fi_min5plusEmb": os.path.basename(OUT_FI_MIN_EMB),
            "fold_models_min5":   [f"xgb_MIN_fold{i}.json" for i in range(1, NFOLDS+1)],
            "fold_models_min5Emb":[f"xgb_MINpEMB_fold{i}.json" for i in range(1, NFOLDS+1)]
        }
    }, f, indent=2, ensure_ascii=False)

print("\n===== 최종 요약 (TEAM-TEST 전용) =====")
print(summary_df)
print("\n저장됨:")
print(" - Summary CSV:", OUT_SUMMARY_CSV)
print(" - Summary JSON:", OUT_SUMMARY_JSON)
print(" - Per-order F1 (teamTEST):", OUT_PER_ORDER)
print(" - Orders (min5):", OUT_ORD_MIN_TT)
print(" - Orders (min5+emb):", OUT_ORD_EMB_TT)
print(" - OOF(min5):", OUT_OOF_MIN)
print(" - OOF(min5+emb):", OUT_OOF_MIN_EMB)
print(" - FI(min5):", OUT_FI_MIN)
print(" - FI(min5+emb):", OUT_FI_MIN_EMB)
print(" - Files @", OUT_DIR)


Mounted at /content/drive
[INFO] device=cuda  tree_method=hist  xgboost=3.0.4
[INFO] rows — prior(train)=20,641,991, teamTEST=11,792,498
[INFO] minimal-5 features: ['order_hour_of_day', 'aisle_id', 'department_id', 'product_id', 'user_id_num']
[WARN] missing embedding — rows 79 — filling 0
[WARN] missing embedding — rows 1,115 — filling 0
[INFO] train sample: 3,000,000 rows (pos=1,768,007)
[xgb_MIN] Fold 1 | AUC=0.64770 | Logloss=0.64113 | BestIter=593
[xgb_MIN] Fold 2 | AUC=0.64880 | Logloss=0.64110 | BestIter=596
[xgb_MIN] Fold 3 | AUC=0.64975 | Logloss=0.64094 | BestIter=590
[thr-search] sample orders: 50,000  rows: 107,930
[thr-search] candidates: 31  (q 0.05~0.95)
[thr-search] 6/31  thr=0.5045  F1=0.63315
[thr-search] 12/31  thr=0.5658  F1=0.60015
[thr-search] 18/31  thr=0.6094  F1=0.54773
[thr-search] 24/31  thr=0.6663  F1=0.47327
[thr-search] 30/31  thr=0.7565  F1=0.36293
[thr-search] 31/31  thr=0.7807  F1=0.33886
[thr-search] best_thr=0.4334  best_f1=0.64142  total=0.7s
[metric

/tmp/ipython-input-4273570480.py:436: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sub_tt_min = df_tt_min.groupby(order_key, group_keys=False).apply(lambda g: to_pred_string(g, "proba_min", thr_min)).reset_index()
/tmp/ipython-input-4273570480.py:442: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sub_tt_me = df_tt_me.groupby(order_key, group_keys=False).apply(lambda g: to_pred_string(g, "proba_min_emb", th

Saved feature importance → /content/drive/MyDrive/data/instacart/perf_min5_vs_min5plusEmb_TEAMTEST_only/fi_min5.csv
Saved feature importance → /content/drive/MyDrive/data/instacart/perf_min5_vs_min5plusEmb_TEAMTEST_only/fi_min5plusEmb.csv

===== 최종 요약 (TEAM-TEST 전용) =====
        split                    model  \
0  OOF(prior)                 XGB_min5   
1  OOF(prior)  XGB_min5_plus_embedding   
2   TEAM-TEST                 XGB_min5   
3   TEAM-TEST  XGB_min5_plus_embedding   

                                       features_used  macro_F1       AUC  \
0  [order_hour_of_day, aisle_id, department_id, p...  0.641422  0.649101   
1  [order_hour_of_day, aisle_id, department_id, p...  0.674872  0.728863   
2                                                NaN  0.680580  0.649293   
3                                                NaN  0.660244  0.732017   

    Logloss  threshold      rows  diag_bestF1  
0  0.640988   0.433414   3000000          NaN  
1  0.594624   0.431078   3000000       